# Segmenting and Clustering Neighborhoods in Toronto - Part Three
## Instructions: Explore and cluster the neighborhoods in Toronto. You can decide to work with only boroughs that contain the word Toronto and then replicate the same analysis we did to the New York City data. It is up to you.

### Just make sure:

1. to add enough Markdown cells to explain what you decided to do and to report any observations you make.
2. to generate maps to visualize your neighborhoods and how they cluster together.

##### Note: Transfered this notebook from IBM Watson to labs.cognitiveclass.ai because IBM Watson popped up a notice that my Lite account was approaching its monthly usage limit and wanted me to open a paid account.


### Install geopy, folium, requests, and beautiful soup

In [6]:
!conda install -c conda-forge geopy --yes 
print("Installed geopy")

!conda install -c conda-forge folium=0.5.0 --yes 
print("Installed folium")

# Added yes when was asked to procced during request install
!conda install beautifulsoup4 --yes
print("Installed beautifulsoup4")

# Added yes when was asked to procced during request install
!conda install requests --yes
print("Installed requests")

print("Installations finished")

Solving environment: / 
The environment is inconsistent, please check the package plan carefully
The following packages are causing the inconsistency:

  - defaults/linux-64::anaconda==5.3.1=py37_0
  - defaults/linux-64::astropy==3.0.4=py37h14c3975_0
  - defaults/linux-64::bkcharts==0.2=py37_0
  - defaults/linux-64::blaze==0.11.3=py37_0
  - defaults/linux-64::bokeh==0.13.0=py37_0
  - defaults/linux-64::bottleneck==1.2.1=py37h035aef0_1
  - defaults/linux-64::dask==0.19.1=py37_0
  - defaults/linux-64::datashape==0.5.4=py37_1
  - defaults/linux-64::mkl-service==1.1.2=py37h90e4bf4_5
  - defaults/linux-64::numba==0.39.0=py37h04863e7_0
  - defaults/linux-64::numexpr==2.6.8=py37hd89afb7_0
  - defaults/linux-64::odo==0.5.1=py37_0
  - defaults/linux-64::pytables==3.4.4=py37ha205bf6_0
  - defaults/linux-64::pytest-arraydiff==0.2=py37h39e3cac_0
  - defaults/linux-64::pytest-astropy==0.4.0=py37_0
  - defaults/linux-64::pytest-doctestplus==0.1.3=py37_0
  - defaults/linux-64::pywavelets==1.0.0=py37h

# Import needed panadas, numpy, and others.

In [62]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

import json # library to handle JSON files
#!conda install -c conda-forge geopy --yes # uncomment this line if you haven't completed the Foursquare API lab
from geopy.geocoders import Nominatim # convert an address into latitude and longitude values

import requests # library to handle requests
from pandas.io.json import json_normalize # tranform JSON file into a pandas dataframe

# Matplotlib and associated plotting modules
import matplotlib.cm as cm
import matplotlib.colors as colors

# import k-means from clustering stage
from sklearn.cluster import KMeans
from sklearn import metrics
from sklearn.metrics import f1_score

import requests

#!conda install -c conda-forge folium=0.5.0 --yes # uncomment this line if you haven't completed the Foursquare API lab
import folium # map rendering library

from bs4 import BeautifulSoup

import random # library for random number generation

from geopy.geocoders import Nominatim # module to convert an address into latitude and longitude values

# libraries for displaying images
from IPython.display import Image 
from IPython.core.display import HTML 

    
print('Libraries imported.')

Libraries imported.


# Set the url

In [8]:
url="https://en.wikipedia.org/wiki/List_of_postal_codes_of_Canada:_M"

# Read in the url

In [9]:
html = requests.get(url).text

# Apply beautiful soup to the html from the url, parse using html

In [10]:
soup=BeautifulSoup(html, "html")

# take first look at the html

In [12]:
# print(soup)

# Use beautiful soup to veiw the data displaying its levels

In [13]:
# print(soup.prettify())

# Using soup, get the table and the TR rows that have the data we need
## Get the 'td' rows from within the 'tr' rows, and create a panda dataframe, naming the columns
## Note: We were instructed to use the American spelling, Neighborhood, instead of the the English spelling in the 'th' row, Neighbourhood
## Remove cases of null postal code, and cases of borough not assigned 

In [22]:
#get the columns
table = soup.find('table',{'class':'wikitable sortable'})
tr_rows = table.find_all('tr')

mk_data = []
for row in tr_rows:
    mk_data.append([td.text.strip() for td in row.find_all('td')])

df = pd.DataFrame(mk_data, columns=['PostalCode', 'Borough', 'Neighborhood'])
df = df[~df['PostalCode'].isnull()] 
df = df[df['Borough'] != 'Not assigned'] 
# df

# Need to handle neighborhooods with unassigned values by assigning borough value

In [23]:
df2=df
df2.loc[df2.Neighborhood == 'Not assigned', 'Neighborhood'] = df2.Borough
# df2

# Combine values of neighborhood per postal code into one column with values separated by columns

In [24]:
# need set(x) to retain having three columns instead of less
df3 = df2.groupby(by=['PostalCode','Borough']).agg(lambda x: ','.join(set(x))).reset_index()

# using df3 instead of print(df3) gives a nicer output
df3


,PostalCode,Borough,Neighborhood
0,M1B,Scarborough,"Malvern,Rouge"
1,M1C,Scarborough,"Highland Creek,Rouge Hill,Port Union"
2,M1E,Scarborough,"Morningside,Guildwood,West Hill"
3,M1G,Scarborough,Woburn
4,M1H,Scarborough,Cedarbrae
5,M1J,Scarborough,Scarborough Village
6,M1K,Scarborough,"Ionview,Kennedy Park,East Birchmount Park"
7,M1L,Scarborough,"Golden Mile,Clairlea,Oakridge"
8,M1M,Scarborough,"Cliffside,Scarborough Village West,Cliffcrest"
9,M1N,Scarborough,"Cliffside West,Birch Cliff"


# Previously for the end of part one, use the .shape method to print the number of rows of your dataframe

In [31]:
# Show last line of part one.
print('Number of rows = ', df3.shape[0])

Number of rows =  103


## As per Instructor's instructions, can use csv file to get latitude and longitude
### Read in the lat long file and look at it

In [26]:
csv_file_lat_long = 'http://cocl.us/Geospatial_data'
df_csv = pd.read_csv(csv_file_lat_long)
df_csv

,Postal Code,Latitude,Longitude
0,M1B,43.806686,-79.194353
1,M1C,43.784535,-79.160497
2,M1E,43.763573,-79.188711
3,M1G,43.770992,-79.216917
4,M1H,43.773136,-79.239476
5,M1J,43.744734,-79.239476
6,M1K,43.727929,-79.262029
7,M1L,43.711112,-79.284577
8,M1M,43.716316,-79.239476
9,M1N,43.692657,-79.264848


### Note that both file have 103 rows, and that the spelling of the PostCode column has a space (i.e. Postal Code) in the lat long file, but not in the first dataframe from the html source.
### Merge the dataframes

In [27]:
df4 = pd.merge(df3, df_csv, left_on='PostalCode', right_on='Postal Code')
df4


,PostalCode,Borough,Neighborhood,Postal Code,Latitude,Longitude
0,M1B,Scarborough,"Malvern,Rouge",M1B,43.806686,-79.194353
1,M1C,Scarborough,"Highland Creek,Rouge Hill,Port Union",M1C,43.784535,-79.160497
2,M1E,Scarborough,"Morningside,Guildwood,West Hill",M1E,43.763573,-79.188711
3,M1G,Scarborough,Woburn,M1G,43.770992,-79.216917
4,M1H,Scarborough,Cedarbrae,M1H,43.773136,-79.239476
5,M1J,Scarborough,Scarborough Village,M1J,43.744734,-79.239476
6,M1K,Scarborough,"Ionview,Kennedy Park,East Birchmount Park",M1K,43.727929,-79.262029
7,M1L,Scarborough,"Golden Mile,Clairlea,Oakridge",M1L,43.711112,-79.284577
8,M1M,Scarborough,"Cliffside,Scarborough Village West,Cliffcrest",M1M,43.716316,-79.239476
9,M1N,Scarborough,"Cliffside West,Birch Cliff",M1N,43.692657,-79.264848


### Drop the second postal code column
#### Incrementing df number instead of in place to allow rerunning the cell

In [30]:
df5=df4.drop('Postal Code', axis=1)
# Show table required for part two.
df5

,PostalCode,Borough,Neighborhood,Latitude,Longitude
0,M1B,Scarborough,"Malvern,Rouge",43.806686,-79.194353
1,M1C,Scarborough,"Highland Creek,Rouge Hill,Port Union",43.784535,-79.160497
2,M1E,Scarborough,"Morningside,Guildwood,West Hill",43.763573,-79.188711
3,M1G,Scarborough,Woburn,43.770992,-79.216917
4,M1H,Scarborough,Cedarbrae,43.773136,-79.239476
5,M1J,Scarborough,Scarborough Village,43.744734,-79.239476
6,M1K,Scarborough,"Ionview,Kennedy Park,East Birchmount Park",43.727929,-79.262029
7,M1L,Scarborough,"Golden Mile,Clairlea,Oakridge",43.711112,-79.284577
8,M1M,Scarborough,"Cliffside,Scarborough Village West,Cliffcrest",43.716316,-79.239476
9,M1N,Scarborough,"Cliffside West,Birch Cliff",43.692657,-79.264848


## End of part two

## Beginning of part three

### As per instructions, it is ok to use only Toronto neighborhoods.
### Get Toronto neighborhoods.

In [32]:
df5.Borough.value_counts()

North York          24
Downtown Toronto    18
Scarborough         17
Etobicoke           12
Central Toronto      9
West Toronto         6
York                 5
East York            5
East Toronto         5
Mississauga          1
Queen's Park         1
Name: Borough, dtype: int64

In [33]:
df6 = df5[(df5.Borough == 'Downtown Toronto') | (df5.Borough == 'Central Toronto')  | \
(df5.Borough == 'West Toronto')  | (df5.Borough == 'East Toronto') ]
df6.Borough.value_counts()

#Downtown Toronto
#Central Toronto
#West Toronto
#East Toronto 
df6

,PostalCode,Borough,Neighborhood,Latitude,Longitude
37,M4E,East Toronto,The Beaches,43.676357,-79.293031
41,M4K,East Toronto,"Riverdale,The Danforth West",43.679557,-79.352188
42,M4L,East Toronto,"The Beaches West,India Bazaar",43.668999,-79.315572
43,M4M,East Toronto,Studio District,43.659526,-79.340923
44,M4N,Central Toronto,Lawrence Park,43.728020,-79.388790
45,M4P,Central Toronto,Davisville North,43.712751,-79.390197
46,M4R,Central Toronto,North Toronto West,43.715383,-79.405678
47,M4S,Central Toronto,Davisville,43.704324,-79.388790
48,M4T,Central Toronto,"Summerhill East,Moore Park",43.689574,-79.383160
49,M4V,Central Toronto,"Summerhill West,Forest Hill SE,Deer Park,South...",43.686412,-79.400049


### Get the latitude and Longitude for Toronto, ON. for centering the map later on.

In [34]:
address = 'Toronto, ON'

geolocator = Nominatim(user_agent="toronto_explorer")
location = geolocator.geocode(address)
latitude = location.latitude
longitude = location.longitude
print('The geograpical coordinate of Toronto are {}, {}.'.format(latitude, longitude))
toronto_latitude = latitude
toronto_longitude = longitude

The geograpical coordinate of Toronto are 43.653963, -79.387207.


### Load the Forsquare iD info to access the Foursquare geo based info.

In [35]:
CLIENT_ID = 'HSW5E4TVOMC3UQFD0KPA0OFY41JHE0KTYLWRUIUVQUWGOKMN' # your Foursquare ID
CLIENT_SECRET = 'HZXQVYWM3E0OMVHKXSOAZ4DBMEFDVVCCVTRKPTA0E5N4PNPP' # your Foursquare Secret
VERSION = '20180604'
LIMIT = 30


In [36]:
def getNearbyVenues(names, borough, latitudes, longitudes, radius=500):
    
    venues_list=[]
    for name, borough, lat, lng in zip(names, borough, latitudes, longitudes):
        print(name)
            
        # create the API request URL
        url = 'https://api.foursquare.com/v2/venues/explore?&client_id={}&client_secret={}&v={}&ll={},{}&radius={}&limit={}'.format(
            CLIENT_ID, 
            CLIENT_SECRET, 
            VERSION, 
            lat, 
            lng, 
            radius, 
            LIMIT)
            
        # make the GET request
        results = requests.get(url).json()["response"]['groups'][0]['items']
        
        # return only relevant information for each nearby venue
        venues_list.append([(
            name, 
            borough, 
            lat, 
            lng, 
            v['venue']['name'], 
            v['venue']['location']['lat'], 
            v['venue']['location']['lng'],  
            v['venue']['categories'][0]['name']) for v in results])

    nearby_venues = pd.DataFrame([item for venue_list in venues_list for item in venue_list])
    nearby_venues.columns = ['Neighborhood', 
                             'Borough',
                  'Neighborhood Latitude', 
                  'Neighborhood Longitude', 
                  'Venue', 
                  'Venue Latitude', 
                  'Venue Longitude', 
                  'Venue Category']
    
    return(nearby_venues)

In [37]:
toronto_venues = getNearbyVenues(names=df6['Neighborhood'],
                                borough= df6['Borough'],
                                 latitudes=df6['Latitude'],
                                 longitudes=df6['Longitude']
                                  )


The Beaches
Riverdale,The Danforth West
The Beaches West,India Bazaar
Studio District
Lawrence Park
Davisville North
North Toronto West
Davisville
Summerhill East,Moore Park
Summerhill West,Forest Hill SE,Deer Park,South Hill,Rathnelly
Rosedale
St. James Town,Cabbagetown
Church and Wellesley
Harbourfront,Regent Park
Ryerson,Garden District
St. James Town
Berczy Park
Central Bay Street
Adelaide,Richmond,King
Harbourfront East,Toronto Islands,Union Station
Design Exchange,Toronto Dominion Centre
Commerce Court,Victoria Hotel
Roselawn
Forest Hill West,Forest Hill North
Yorkville,The Annex,North Midtown
University of Toronto,Harbord
Kensington Market,Grange Park,Chinatown
South Niagara,Bathurst Quay,King and Spadina,Railway Lands,CN Tower,Harbourfront West,Island airport
Stn A PO Boxes 25 The Esplanade
Underground city,First Canadian Place
Christie
Dovercourt Village,Dufferin
Little Portugal,Trinity
Exhibition Place,Parkdale Village,Brockton
High Park,The Junction South
Parkdale,Roncesvall

In [38]:
print(toronto_venues.shape)
print(toronto_venues.head())
toronto_venues.groupby('Neighborhood').count()

(838, 8)
                  Neighborhood       Borough  Neighborhood Latitude  \
0                  The Beaches  East Toronto              43.676357   
1                  The Beaches  East Toronto              43.676357   
2                  The Beaches  East Toronto              43.676357   
3                  The Beaches  East Toronto              43.676357   
4  Riverdale,The Danforth West  East Toronto              43.679557   

   Neighborhood Longitude                               Venue  Venue Latitude  \
0              -79.293031  The Big Carrot Natural Food Market       43.678879   
1              -79.293031                 Grover Pub and Grub       43.679181   
2              -79.293031                           Starbucks       43.678798   
3              -79.293031                       Upper Beaches       43.680563   
4              -79.352188                            Pantheon       43.677621   

   Venue Longitude     Venue Category  
0       -79.297734  Health Food Store

,Borough,Neighborhood Latitude,Neighborhood Longitude,Venue,Venue Latitude,Venue Longitude,Venue Category
Neighborhood,,,,,,,
"Adelaide,Richmond,King",30,30,30,30,30,30,30
Berczy Park,30,30,30,30,30,30,30
Business Reply Mail Processing Centre 969 Eastern,17,17,17,17,17,17,17
Central Bay Street,30,30,30,30,30,30,30
Christie,16,16,16,16,16,16,16
Church and Wellesley,30,30,30,30,30,30,30
"Commerce Court,Victoria Hotel",30,30,30,30,30,30,30
Davisville,30,30,30,30,30,30,30
Davisville North,11,11,11,11,11,11,11


In [39]:
print('There are {} unique Neighborhoods.'.format(len(df6['Neighborhood'].unique())))

There are 38 uniques categories.


In [42]:
print('There are {} unique combinations of Neighborhood, Bororugh.'.format(len(df6.groupby(['Neighborhood', 'Borough']).size())))

There are 38 uniques combinations of Neighborhood, Bororugh.


In [43]:
print('There are {} unique venue categories.'.format(len(toronto_venues['Venue Category'].unique())))

There are 192 unique venue categories.


# One of the values of Venue Category is Neighborhood, this causes get_dummies to make a variable called Neighborhood.  There is already a column called Neighborhood, so this causes there to be TWO columns with the same name: Neighborhood, in the same Dataframe.  This later prevents calucualtions using groupby.
## Changing 'Venue Category' values of Neighborhood to 'Neighborhood Category', so that there will not be two columns with the same name.

In [44]:

#toronto_venues.head()
toronto_venues.loc[toronto_venues['Venue Category'] == 'Neighborhood', 'Venue Category'] = 'Neighborhood Category'
test1 =toronto_venues[(toronto_venues['Venue Category'] == 'Neighborhood') | \
                       (toronto_venues['Venue Category'] == 'Neighborhood Category' )]
test1.head()


,Neighborhood,Borough,Neighborhood Latitude,Neighborhood Longitude,Venue,Venue Latitude,Venue Longitude,Venue Category
3,The Beaches,East Toronto,43.676357,-79.293031,Upper Beaches,43.680563,-79.292869,Neighborhood Category
64,Studio District,East Toronto,43.659526,-79.340923,Leslieville,43.662070,-79.337856,Neighborhood Category
399,"Adelaide,Richmond,King",Downtown Toronto,43.650571,-79.384568,Downtown Toronto,43.653232,-79.385296,Neighborhood Category
412,"Harbourfront East,Toronto Islands,Union Station",Downtown Toronto,43.640816,-79.381752,Harbourfront,43.639526,-79.380688,Neighborhood Category


## Convert categorical columns into nueric dummy variable columns for use with cluster analysis.
### Note: Neighborhood is the neighborhood but it is also a category, so it becomes two columns with the same name, and later cannot groupby; therefore, was changed above.

In [45]:
dummies = pd.get_dummies(toronto_venues['Venue Category'])
dummies2 = pd.concat([toronto_venues['Neighborhood'], dummies], axis=1)
print(dummies2.columns)
print(dummies2.shape)                

Index(['Neighborhood', 'Adult Boutique', 'Airport', 'Airport Food Court',
       'Airport Gate', 'Airport Lounge', 'Airport Service', 'Airport Terminal',
       'American Restaurant', 'Antique Shop',
       ...
       'Thrift / Vintage Store', 'Toy / Game Store', 'Trail', 'Train Station',
       'Vegetarian / Vegan Restaurant', 'Video Game Store',
       'Vietnamese Restaurant', 'Wine Bar', 'Wings Joint', 'Yoga Studio'],
      dtype='object', length=193)
(838, 193)


### Group the dummy columns by Neighborhood, get the mean to show relative frequency of venue category within a neighborhood.

In [48]:

dummy_means = dummies2.groupby('Neighborhood').mean().reset_index()
print('dummy_means shape:' ,dummy_means.shape)
dummy_means

dummy_means shape: (38, 193)


,Neighborhood,Adult Boutique,Airport,Airport Food Court,Airport Gate,Airport Lounge,Airport Service,Airport Terminal,American Restaurant,Antique Shop,Aquarium,Art Gallery,Art Museum,Arts & Crafts Store,Asian Restaurant,Athletics & Sports,Auto Workshop,BBQ Joint,Baby Store,Bagel Shop,Bakery,Bank,Bar,Basketball Stadium,Beer Bar,Beer Store,Belgian Restaurant,Bistro,Boat or Ferry,Bookstore,Boutique,Breakfast Spot,Brewery,Bubble Tea Shop,Burger Joint,Burrito Place,Bus Line,Butcher,Café,Cajun / Creole Restaurant,Caribbean Restaurant,Cheese Shop,Chinese Restaurant,Chocolate Shop,Church,Climbing Gym,Clothing Store,Cocktail Bar,Coffee Shop,College Arts Building,College Gym,Comfort Food Restaurant,Comic Shop,Concert Hall,Convenience Store,Cosmetics Shop,Coworking Space,Creperie,Cuban Restaurant,Dance Studio,Deli / Bodega,Dessert Shop,Diner,Discount Store,Dog Run,Dumpling Restaurant,Eastern European Restaurant,Ethiopian Restaurant,Falafel Restaurant,Farmers Market,Fast Food Restaurant,Fish & Chips Shop,Fish Market,Flea Market,Flower Shop,Food,Food & Drink Shop,Food Court,Food Truck,Fountain,French Restaurant,Fried Chicken Joint,Fruit & Vegetable Store,Furniture / Home Store,Garden,Garden Center,Gastropub,Gay Bar,General Entertainment,General Travel,Gift Shop,Gourmet Shop,Greek Restaurant,Grocery Store,Gym,Gym / Fitness Center,Harbor / Marina,Health Food Store,Historic Site,History Museum,Hobby Shop,Hotel,Hotel Bar,Ice Cream Shop,Indian Restaurant,Indie Movie Theater,Intersection,Italian Restaurant,Japanese Restaurant,Jazz Club,Jewelry Store,Jewish Restaurant,Juice Bar,Korean Restaurant,Lake,Latin American Restaurant,Light Rail Station,Liquor Store,Lounge,Market,Medical Center,Men's Store,Metro Station,Mexican Restaurant,Middle Eastern Restaurant,Miscellaneous Shop,Modern European Restaurant,Monument / Landmark,Movie Theater,Museum,Music Store,Music Venue,Neighborhood Category,New American Restaurant,Nightclub,Noodle House,Office,Opera House,Organic Grocery,Park,Performing Arts Venue,Pet Store,Pharmacy,Pizza Place,Plane,Playground,Plaza,Poke Place,Pool,Portuguese Restaurant,Pub,Ramen Restaurant,Record Shop,Recording Studio,Rental Car Location,Restaurant,Salad Place,Salon / Barbershop,Sandwich Place,Sculpture Garden,Seafood Restaurant,Skate Park,Skating Rink,Smoke Shop,Smoothie Shop,Spa,Speakeasy,Sporting Goods Shop,Sports Bar,Stadium,Stationery Store,Steakhouse,Supermarket,Sushi Restaurant,Swim School,Taco Place,Tailor Shop,Taiwanese Restaurant,Tea Room,Tennis Court,Thai Restaurant,Theater,Theme Restaurant,Thrift / Vintage Store,Toy / Game Store,Trail,Train Station,Vegetarian / Vegan Restaurant,Video Game Store,Vietnamese Restaurant,Wine Bar,Wings Joint,Yoga Studio
0,"Adelaide,Richmond,King",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.033333,0.00,0.000000,0.000000,0.000000,0.000000,0.066667,0.0000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.066667,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.033333,0.000000,0.000000,0.033333,0.000000,0.000000,0.033333,0.000000,0.000000,0.033333,0.000000,0.00,0.000000,0.00,0.000000,0.066667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,0.033333,0.000000,0.0000,0.033333,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,0.000000,0.033333,0.000000,0.000000,0.033333

# Show top five venue categories per neighborhood

In [202]:
num_top_venues = 5

for hood in dummy_means['Neighborhood']:
    print("----"+hood+"----")
    temp = dummy_means[dummy_means['Neighborhood'] == hood].T.reset_index()
    temp.columns = ['venue','freq']
    temp = temp.iloc[1:]
    temp['freq'] = temp['freq'].astype(float)
    temp = temp.round({'freq': 2})
    print(temp.sort_values('freq', ascending=False).reset_index(drop=True).head(num_top_venues))
    print('\n')

----Adelaide,Richmond,King----
              venue  freq
0        Steakhouse  0.10
1             Hotel  0.07
2              Café  0.07
3  Asian Restaurant  0.07
4       Coffee Shop  0.03


----Bathurst Quay,King and Spadina,CN Tower,Island airport,Railway Lands,South Niagara,Harbourfront West----
              venue  freq
0    Airport Lounge  0.15
1   Airport Service  0.15
2  Airport Terminal  0.15
3   Harbor / Marina  0.08
4           Airport  0.08


----Berczy Park----
                venue  freq
0  Seafood Restaurant  0.07
1        Cocktail Bar  0.07
2              Bakery  0.07
3                Café  0.07
4      Farmers Market  0.07


----Brockton,Exhibition Place,Parkdale Village----
               venue  freq
0     Breakfast Spot  0.10
1               Café  0.10
2        Coffee Shop  0.10
3                Bar  0.05
4  Convenience Store  0.05


----Business Reply Mail Processing Centre 969 Eastern----
                venue  freq
0  Light Rail Station  0.12
1         Pizza Place  0.

### Put into a pandas dataframe
### Step 1 Write a function to sort the venues in descending order.

In [49]:
def return_most_common_venues(row, num_top_venues):
    row_categories = row.iloc[1:]
    row_categories_sorted = row_categories.sort_values(ascending=False)
    
    return row_categories_sorted.index.values[0:num_top_venues]


### Step 2 Create the new dataframe and display the top 10 venues for each neighborhood.

In [131]:
num_top_venues = 30

indicators = ['st', 'nd', 'rd']

# create columns according to number of top venues
columns = ['Neighborhood']
for ind in np.arange(num_top_venues):
    try:
        columns.append('{}{} Most Common Venue'.format(ind+1, indicators[ind]))
    except:
        columns.append('{}th Most Common Venue'.format(ind+1))

# create a new dataframe
neighborhoods_venues_sorted = pd.DataFrame(columns=columns)
neighborhoods_venues_sorted['Neighborhood'] = dummy_means['Neighborhood']

for ind in np.arange(dummy_means.shape[0]):
    neighborhoods_venues_sorted.iloc[ind, 1:] = return_most_common_venues(dummy_means.iloc[ind, :], num_top_venues)

neighborhoods_venues_sorted.head()

,Neighborhood,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
0,"Adelaide,Richmond,King",Steakhouse,Café,Asian Restaurant,Hotel,Concert Hall,Monument / Landmark,Pizza Place,Plaza,Sushi Restaurant,Lounge,Seafood Restaurant,Neighborhood Category,Speakeasy,Bar,Opera House,Noodle House,General Travel,Smoke Shop,Coffee Shop,Greek Restaurant,Gym / Fitness Center,American Restaurant,Vegetarian / Vegan Restaurant,Food Court,Gastropub,Comic Shop,Comfort Food Restaurant,Garden Center,Convenience Store,College Gym
1,Berczy Park,Cocktail Bar,Café,Seafood Restaurant,Farmers Market,Belgian Restaurant,Breakfast Spot,Bakery,Basketball Stadium,Beer Bar,Concert Hall,Bistro,Restaurant,Steakhouse,Liquor Store,Pub,Park,Clothing Store,Coffee Shop,Museum,Jazz Club,Comfort Food Restaurant,Italian Restaurant,Vegetarian / Vegan Restaurant,French Restaurant,Thai Restaurant,Tea Room,Gastropub,Gay Bar,Garden Center,Cosmetics Shop
2,Business Reply Mail Processing Centre 969 Eastern,Light Rail Station,Spa,Burrito Place,Restaurant,Auto Workshop,Skate Park,Fast Food Restaurant,Farmers Market,Brewery,Garden,Garden Center,Pizza Place,Comic Shop,Smoke Shop,Park,Recording Studio,Discount Store,Dessert Shop,Diner,Dog Run,Dance Studio,Deli / Bodega,Cuban Restaurant,Creperie,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Comfort Food Restaurant,College Gym
3,Central Bay Street,Coffee Shop,Italian Restaurant,Spa,Bubble Tea Shop,Pizza Place,Art Museum,Portuguese Restaurant,Steakhouse,Bar,Sushi Restaurant,Ramen Restaurant,Miscellaneous Shop,Tea Room,Japanese Restaurant,Modern European Restaurant,Chinese Restaurant,Park,Seafood Restaurant,Vegetarian / Vegan Restaurant,Sandwich Place,Gastropub,Dance Studio,Deli / Bodega,Dessert Shop,Yoga Studio,Creperie,Cuban Restaurant,Discount Store,Coworking Space,Cosmetics Shop
4,Christie,Grocery Store,Café,Park,Athletics & Sports,Italian Restaurant,Diner,Nightclub,Convenience Store,Restaurant,Baby Store,Coffee Shop,Art Gallery,Discount Store,Fish & Chips Shop,Fast Food Restaurant,Farmers Market,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Airport Food Court,Flea Market,Dessert Shop,Deli / Bodega,Dance Studio,Cuban Restaurant,Creperie,Coworking Space,Airport Gate


## Cluster the neighborhoods
### Run *k*-means to cluster the neighborhoods.
### At first, found majority of neighborhoods in one cluster, with few neighborhoods in other clusters.
### To address this issue, tried changing number of clusters, max_iter, n_init and init method.
### After increasing the number of top venues used from 10 to 30, with 6 clusters, init of random, increased n_init and max_iter, was able to get at least two clusters with more than a few neighborhoods.

In [132]:
# set number of clusters
kclusters = 6

dummy_means_clustering = dummy_means.drop('Neighborhood', 1)
print(dummy_means_clustering.columns)

# run k-means clustering
kmeans = KMeans(n_clusters=kclusters, random_state=0,max_iter=1000,n_init=100,init='random').fit(dummy_means_clustering)

# kmeans - to show defaults

# check cluster labels generated for each row in the dataframe
kmeans.labels_[0:30] 

print(kmeans.labels_)


Index(['Adult Boutique', 'Airport', 'Airport Food Court', 'Airport Gate',
       'Airport Lounge', 'Airport Service', 'Airport Terminal',
       'American Restaurant', 'Antique Shop', 'Aquarium',
       ...
       'Thrift / Vintage Store', 'Toy / Game Store', 'Trail', 'Train Station',
       'Vegetarian / Vegan Restaurant', 'Video Game Store',
       'Vietnamese Restaurant', 'Wine Bar', 'Wings Joint', 'Yoga Studio'],
      dtype='object', length=192)
[4 1 1 4 4 1 4 4 1 4 1 4 2 1 4 4 4 2 1 4 4 1 2 3 4 1 4 4 1 4 5 0 4 0 1 4 4
 4]


### Create a new dataframe that includes the cluster as well as the top 10 venues for each neighborhood.

In [133]:
# add clustering labels
# Create new df to allow rerunning

neighborhoods_venues_sorted.insert(0, 'Cluster Labels', kmeans.labels_)

print('neighborhoods_venues_sorted =', neighborhoods_venues_sorted.shape)

print(neighborhoods_venues_sorted.columns)
cluster_numbers = neighborhoods_venues_sorted.groupby('Cluster Labels').count().reset_index()
print(cluster_numbers)

data_merged = df6

print('df6 shape =', df6.shape)
data_merged = df6

# merge toronto_grouped with toronto_data to add latitude/longitude for each neighborhood
data_merged = data_merged.join(neighborhoods_venues_sorted.set_index('Neighborhood'), on='Neighborhood')

#print(data_merged['Cluster Labels'].value_counts)


print('data_merged =', data_merged.shape)
data_merged.head() # check the last columns!

data_merged.groupby('Cluster Labels').count()

neighborhoods_venues_sorted = (38, 32)
Index(['Cluster Labels', 'Neighborhood', '1st Most Common Venue',
       '2nd Most Common Venue', '3rd Most Common Venue',
       '4th Most Common Venue', '5th Most Common Venue',
       '6th Most Common Venue', '7th Most Common Venue',
       '8th Most Common Venue', '9th Most Common Venue',
       '10th Most Common Venue', '11th Most Common Venue',
       '12th Most Common Venue', '13th Most Common Venue',
       '14th Most Common Venue', '15th Most Common Venue',
       '16th Most Common Venue', '17th Most Common Venue',
       '18th Most Common Venue', '19th Most Common Venue',
       '20th Most Common Venue', '21th Most Common Venue',
       '22th Most Common Venue', '23th Most Common Venue',
       '24th Most Common Venue', '25th Most Common Venue',
       '26th Most Common Venue', '27th Most Common Venue',
       '28th Most Common Venue', '29th Most Common Venue',
       '30th Most Common Venue'],
      dtype='object')
   Cluster Labels  Ne

,PostalCode,Borough,Neighborhood,Latitude,Longitude,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
Cluster Labels,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
1,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11
2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
4,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
5,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


### Create map to visualize clusters

In [140]:
# create map
map_clusters = folium.Map(location=[toronto_latitude, toronto_longitude], zoom_start=11)

# set color scheme for the clusters
x = np.arange(kclusters)
ys = [i + x + (i*x)**2 for i in range(kclusters)]
colors_array = cm.rainbow(np.linspace(0, 1, len(ys)))
rainbow = [colors.rgb2hex(i) for i in colors_array]

# add markers to the map
markers_colors = []
for lat, lon, poi, cluster in zip(data_merged['Latitude'], data_merged['Longitude'], data_merged['Neighborhood'], \
                                  data_merged['Cluster Labels']):
    label = folium.Popup(str(poi) + ' Cluster ' + str(cluster), parse_html=True)
    folium.CircleMarker(
        [lat, lon],
        radius=5,
        popup=label,
        color=rainbow[cluster-1],
        fill=True,
        fill_color=rainbow[cluster-1],
        fill_opacity=0.7).add_to(map_clusters)
       
map_clusters

### Showing columns in the final dataframe

In [92]:
data_merged.columns

Index(['PostalCode', 'Borough', 'Neighborhood', 'Latitude', 'Longitude',
       'Cluster Labels', '1st Most Common Venue', '2nd Most Common Venue',
       '3rd Most Common Venue', '4th Most Common Venue',
       '5th Most Common Venue', '6th Most Common Venue',
       '7th Most Common Venue', '8th Most Common Venue',
       '9th Most Common Venue', '10th Most Common Venue',
       '11th Most Common Venue', '12th Most Common Venue',
       '13th Most Common Venue', '14th Most Common Venue',
       '15th Most Common Venue', '16th Most Common Venue',
       '17th Most Common Venue', '18th Most Common Venue',
       '19th Most Common Venue', '20th Most Common Venue',
       '21th Most Common Venue', '22th Most Common Venue',
       '23th Most Common Venue', '24th Most Common Venue',
       '25th Most Common Venue', '26th Most Common Venue',
       '27th Most Common Venue', '28th Most Common Venue',
       '29th Most Common Venue', '30th Most Common Venue'],
      dtype='object')

# Show each cluster with its most common venue categories

# Cluster 1

In [93]:
data_merged.loc[data_merged['Cluster Labels'] == 0, data_merged.columns[[2] + list(range(5, data_merged.shape[1]))]]

,Neighborhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
37,The Beaches,0,Coffee Shop,Health Food Store,Pub,Neighborhood Category,Dance Studio,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Discount Store,Diner,Dessert Shop,Deli / Bodega,Yoga Studio,Farmers Market,Creperie,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Comic Shop,Comfort Food Restaurant,College Gym,Cuban Restaurant,Fish & Chips Shop,Fast Food Restaurant,Furniture / Home Store,Greek Restaurant,Gourmet Shop
49,"Summerhill West,Forest Hill SE,Deer Park,South...",0,Pub,Coffee Shop,American Restaurant,Medical Center,Sports Bar,Bagel Shop,Supermarket,Sushi Restaurant,Fried Chicken Joint,Light Rail Station,Pizza Place,Convenience Store,Vietnamese Restaurant,Diner,Eastern European Restaurant,College Gym,Dumpling Restaurant,Dog Run,Comfort Food Restaurant,Discount Store,Comic Shop,Cosmetics Shop,Coworking Space,Concert Hall,Dessert Shop,Deli / Bodega,Dance Studio,Ethiopian Restaurant,Cuban Restaurant,Creperie


# Cluster 2

In [94]:
data_merged.loc[data_merged['Cluster Labels'] == 1, data_merged.columns[[2] + list(range(5, data_merged.shape[1]))]]

,Neighborhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
41,"Riverdale,The Danforth West",1,Greek Restaurant,Italian Restaurant,Ice Cream Shop,Yoga Studio,Cosmetics Shop,Pizza Place,Dessert Shop,Diner,Pub,Restaurant,Bubble Tea Shop,Brewery,Bookstore,Juice Bar,Bakery,Japanese Restaurant,Spa,Indian Restaurant,Grocery Store,Trail,Fruit & Vegetable Store,Food Truck,Deli / Bodega,Dumpling Restaurant,Dog Run,Discount Store,Gastropub,Gay Bar,Dance Studio,Garden Center
42,"The Beaches West,India Bazaar",1,Sandwich Place,Pet Store,Sushi Restaurant,Steakhouse,Fish & Chips Shop,Fast Food Restaurant,Light Rail Station,Liquor Store,Brewery,Intersection,Burger Joint,Burrito Place,Pub,Movie Theater,Pizza Place,Park,Italian Restaurant,Coffee Shop,Food & Drink Shop,Ice Cream Shop,Gym,Garden Center,Diner,Dessert Shop,Deli / Bodega,Dance Studio,Cuban Restaurant,Creperie,Food,Garden
45,Davisville North,1,Pizza Place,Asian Restaurant,Gym,Clothing Store,Dance Studio,Burger Joint,Sandwich Place,Breakfast Spot,Food & Drink Shop,Park,Hotel,Cuban Restaurant,Dog Run,General Entertainment,General Travel,Discount Store,Diner,Dessert Shop,Deli / Bodega,Gift Shop,Creperie,Garden,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Gourmet Shop,Comic Shop,Greek Restaurant,Comfort Food Restaurant
52,Church and Wellesley,1,Gay Bar,Creperie,Salon / Barbershop,Restaurant,Bubble Tea Shop,Burger Joint,Ramen Restaurant,Pub,Pizza Place,General Entertainment,Park,Coffee Shop,Dance Studio,Bookstore,Mexican Restaurant,Men's Store,Diner,Ethiopian Restaurant,Juice Bar,Japanese Restaurant,Ice Cream Shop,Hobby Shop,Wings Joint,Gastropub,Breakfast Spot,Adult Boutique,Theme Restaurant,Vietnamese Restaurant,Tea Room,Basketball Stadium
56,Berczy Park,1,Cocktail Bar,Café,Seafood Restaurant,Farmers Market,Belgian Restaurant,Breakfast Spot,Bakery,Basketball Stadium,Beer Bar,Concert Hall,Bistro,Restaurant,Steakhouse,Liquor Store,Pub,Park,Clothing Store,Coffee Shop,Museum,Jazz Club,Comfort Food Restaurant,Italian Restaurant,Vegetarian / Vegan Restaurant,French Restaurant,Thai Restaurant,Tea Room,Gastropub,Gay Bar,Garden Center,Cosmetics Shop
59,"Harbourfront East,Toronto Islands,Union Station",1,Park,Hotel,Café,Performing Arts Venue,Salad Place,Sports Bar,Sporting Goods Shop,Basketball Stadium,Japanese Restaurant,Bistro,Lake,Lounge,Bubble Tea Shop,Bakery,Deli / Bodega,Plaza,Dance Studio,Chinese Restaurant,Office,New American Restaurant,Italian Restaurant,Skating Rink,Aquarium,Ice Cream Shop,Supermarket,Art Gallery,Neighborhood Category,Flower Shop,Cosmetics Shop,French Restaurant
68,"South Niagara,Bathurst Quay,King and Spadina,R...",1,Airport Service,Airport Terminal,Airport Lounge,Harbor / Marina,Sculpture Garden,Boat or Ferry,Plane,Airport Gate,Airport Food Court,Airport,Discount Store,Fast Food Restaurant,Farmers Market,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Diner,Fish Market,Dessert Shop,Deli / Bodega,Dance Studio,Cuban Restaurant,Creperie,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Fish & Chips Shop
69,Stn A PO Boxes 25 The Esplanade,1,Cocktail Bar,Café,Restaurant,Seafood Restaurant,Farmers Market,Beer Bar,Food Truck,Coffee Shop,Clothing Store,Park,Concert Hall,Museum,Breakfast Spot,Belgian Restaurant,Jazz Club,Steakhouse,Comfort Food Restaurant,Tailor 

# Cluster 3

In [95]:
data_merged.loc[data_merged['Cluster Labels'] == 2, data_merged.columns[[2] + list(range(5, data_merged.shape[1]))]]

,Neighborhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
44,Lawrence Park,2,Bus Line,Park,Swim School,Deli / Bodega,Farmers Market,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Discount Store,Diner,Dessert Shop,Yoga Studio,Fish & Chips Shop,Dance Studio,Cuban Restaurant,Creperie,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Comic Shop,Comfort Food Restaurant,Fast Food Restaurant,Fish Market,College Arts Building,Garden,Grocery Store,Greek Restaurant
50,Rosedale,2,Park,Playground,Trail,Yoga Studio,Dance Studio,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Discount Store,Diner,Dessert Shop,Deli / Bodega,Creperie,Cuban Restaurant,Fast Food Restaurant,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Comic Shop,Comfort Food Restaurant,College Gym,Farmers Market,Fish & Chips Shop,Coffee Shop,Furniture / Home Store,Greek Restaurant,Gourmet Shop
64,"Forest Hill West,Forest Hill North",2,Park,Trail,Sushi Restaurant,Jewelry Store,Yoga Studio,Deli / Bodega,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Discount Store,Diner,Dessert Shop,Dance Studio,Fast Food Restaurant,Cuban Restaurant,Creperie,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Comic Shop,Comfort Food Restaurant,College Gym,Farmers Market,Fish Market,Fish & Chips Shop,Furniture / Home Store,Greek Restaurant


# Cluster 4

In [96]:
data_merged.loc[data_merged['Cluster Labels'] == 3, data_merged.columns[[2] + list(range(5, data_merged.shape[1]))]]

,Neighborhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
63,Roselawn,3,Garden,Fish & Chips Shop,Farmers Market,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Discount Store,Diner,Dessert Shop,Deli / Bodega,Yoga Studio,Dance Studio,Cuban Restaurant,Creperie,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Comic Shop,Comfort Food Restaurant,College Gym,Fast Food Restaurant,Fish Market,Wings Joint,Garden Center,Gym,Grocery Store,Greek Restaurant


# Cluster 5

In [97]:
data_merged.loc[data_merged['Cluster Labels'] == 4, data_merged.columns[[2] + list(range(5, data_merged.shape[1]))]]

,Neighborhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
43,Studio District,4,Café,Coffee Shop,Italian Restaurant,American Restaurant,Bakery,Yoga Studio,Sandwich Place,Fish Market,Latin American Restaurant,Bookstore,Seafood Restaurant,Cheese Shop,Middle Eastern Restaurant,Chinese Restaurant,Music Store,Coworking Space,Neighborhood Category,Park,Stationery Store,Comfort Food Restaurant,Ice Cream Shop,Gastropub,Food,Cuban Restaurant,Fruit & Vegetable Store,Deli / Bodega,Dance Studio,Furniture / Home Store,Garden,Creperie
46,North Toronto West,4,Sporting Goods Shop,Coffee Shop,Yoga Studio,Fast Food Restaurant,Diner,Metro Station,Mexican Restaurant,Dessert Shop,Park,Clothing Store,Pet Store,Chinese Restaurant,Rental Car Location,Salon / Barbershop,Sandwich Place,Spa,Gift Shop,BBQ Joint,Airport,Falafel Restaurant,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Discount Store,Airport Food Court,Airport Gate,Airport Lounge,Airport Service,Deli / Bodega
47,Davisville,4,Dessert Shop,Sandwich Place,Pizza Place,Coffee Shop,Italian Restaurant,Café,Sushi Restaurant,Farmers Market,Seafood Restaurant,Restaurant,Brewery,Diner,Flower Shop,Indian Restaurant,Park,Gourmet Shop,Thai Restaurant,Pharmacy,Greek Restaurant,Gym,Toy / Game Store,Convenience Store,Discount Store,College Gym,Comfort Food Restaurant,Dance Studio,Deli / Bodega,Cosmetics Shop,Cuban Restaurant,Comic Shop
51,"St. James Town,Cabbagetown",4,Restaurant,Coffee Shop,Italian Restaurant,Café,Pet Store,Beer Store,Butcher,Pub,Caribbean Restaurant,Playground,Park,Deli / Bodega,Bakery,Diner,Market,Liquor Store,Jewelry Store,Japanese Restaurant,Indian Restaurant,Gastropub,General Entertainment,Bank,Gift Shop,Thai Restaurant,Taiwanese Restaurant,Bagel Shop,Eastern European Restaurant,Fish Market,Fish & Chips Shop,Fast Food Restaurant
53,"Harbourfront,Regent Park",4,Coffee Shop,Park,Bakery,Pub,Café,Mexican Restaurant,Breakfast Spot,Yoga Studio,French Restaurant,Performing Arts Venue,Chocolate Shop,Dessert Shop,Restaurant,Farmers Market,Spa,Gym / Fitness Center,Historic Site,Fountain,Creperie,Garden Center,Gastropub,Deli / Bodega,Gay Bar,Dance Studio,Cuban Restaurant,Coworking Space,Food Truck,General Entertainment,Cosmetics Shop,Convenience Store
54,"Ryerson,Garden District",4,Café,Coffee Shop,Clothing Store,Beer Bar,Taco Place,Sandwich Place,Diner,Music Venue,Ramen Restaurant,Burger Joint,Japanese Restaurant,Burrito Place,Plaza,Steakhouse,Pizza Place,Mexican Restaurant,Movie Theater,Gastropub,Tea Room,Comic Shop,Thai Restaurant,Theater,Vegetarian / Vegan Restaurant,Art Gallery,American Restaurant,Concert Hall,Coworking Space,Dessert Shop,Deli / Bodega,Convenience Store
55,St. James Town,4,Coffee Shop,Gastropub,Italian Restaurant,Restaurant,Hotel,Japanese Restaurant,Creperie,Diner,Middle Eastern Restaurant,BBQ Joint,Food Truck,Poke Place,Spa,Theater,Church,Cosmetics Shop,American Restaurant,Performing Arts Venue,Gym,Café,Discount Store,Dessert Shop,Dog Run,Dumpling Restaurant,Cuban Restaurant,Deli / Bodega,Dance Studio,Ethiopian Restaurant,Coworking Space,Convenience Store
57,Central Bay Street,4,Coffee Shop,Italian Restaurant,Spa,Bubble Tea Shop,Pizza Place,Art Museum,Portuguese Restaurant,Steakhouse,Bar,Sushi Restaurant,Ramen Restaurant,Miscellaneous Shop,Tea Room,Japanese Restaurant,Modern European Restaurant,Chinese R

# Cluster 6

In [136]:
data_merged.loc[data_merged['Cluster Labels'] == 5, data_merged.columns[[2] + list(range(5, data_merged.shape[1]))]]

,Neighborhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue,11th Most Common Venue,12th Most Common Venue,13th Most Common Venue,14th Most Common Venue,15th Most Common Venue,16th Most Common Venue,17th Most Common Venue,18th Most Common Venue,19th Most Common Venue,20th Most Common Venue,21th Most Common Venue,22th Most Common Venue,23th Most Common Venue,24th Most Common Venue,25th Most Common Venue,26th Most Common Venue,27th Most Common Venue,28th Most Common Venue,29th Most Common Venue,30th Most Common Venue
48,"Summerhill East,Moore Park",5,Restaurant,Gym,Playground,Tennis Court,Yoga Studio,Ethiopian Restaurant,Eastern European Restaurant,Dumpling Restaurant,Dog Run,Discount Store,Diner,Dessert Shop,Deli / Bodega,Dance Studio,Cuban Restaurant,Farmers Market,Creperie,Coworking Space,Cosmetics Shop,Convenience Store,Concert Hall,Comic Shop,Comfort Food Restaurant,College Gym,Falafel Restaurant,Fast Food Restaurant,Coffee Shop,Fish & Chips Shop,Greek Restaurant,Gourmet Shop


# Possible labels for clusters:
## 1 - Neighborhood/Medical Center with coffee and pub
## 2 - Urban mix (hotel, rail, airport, shopping, bars)
## 3 - Parks
## 4 - Garden
## 5 - Cafe - restaurant
## 6 - Gym (playground, gym, tennis, yoga, dog run)